# Stage 1 — 09B: real-only grouped V-JEPA 2.1-B probe CV

The public A7 anchor is retained as the reference, but this experiment rebuilds
the **attentive probe** using only real DLC samples and group-aware folds.

Protocol:

1. Official V-JEPA 2.1-B encoder, frozen.
2. Fresh attentive probe per fold; **no A7 probe warm-start** for OOF integrity.
3. `StratifiedGroupKFold` on DLC train, grouping by `group`.
4. Save true OOF probabilities on the 483 training videos.
5. Evaluate every fold model on the untouched fixed DLC val.
6. Mean fold probabilities to form a low-variance probe ensemble.

The OOF predictions are used by notebook 11 as a real-domain proxy for fusion
selection, instead of tuning on the already-saturated 104-video val split.


## 1. Setup


In [ ]:
from __future__ import annotations

import copy
import gc
import json
import math
import os
import subprocess
import sys
import time
from pathlib import Path

REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage1-sangchun"

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)
except ModuleNotFoundError:
    print("Not running in Google Colab; Drive mount skipped.")

if IN_COLAB:
    REPO_ROOT = Path("/content/Blackbox-Detection")
    if not (REPO_ROOT / ".git").is_dir():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", BRANCH,
            "--single-branch", REPO_URL, str(REPO_ROOT),
        ], check=True)
    else:
        current_branch = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "branch", "--show-current"],
            check=True, capture_output=True, text=True,
        ).stdout.strip()
        if current_branch != BRANCH:
            subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", BRANCH], check=True)
        dirty = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "status", "--porcelain"],
            check=True, capture_output=True, text=True,
        ).stdout.strip()
        if dirty:
            print("WARNING: local repo has changes; git pull skipped.")
        else:
            subprocess.run(
                ["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", BRANCH],
                check=True,
            )
else:
    REPO_ROOT = Path.cwd().resolve()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / "pyproject.toml").is_file():
        raise FileNotFoundError("Run this notebook inside Blackbox-Detection repository.")

os.chdir(REPO_ROOT)

# Keep Colab's binary scientific stack intact. Install only the Stage 1 extras
# and then this repository editable with --no-deps, matching the current notebooks.
COLAB_EXTRAS = [
    "av>=15,<17", "timm==1.0.15", "fvcore==0.1.5.post20221221",
    "iopath==0.1.10", "yacs==0.1.8", "einops==0.8.1",
    "omegaconf==2.3.0", "hydra-core==1.3.2", "easydict==1.13",
]
if IN_COLAB:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed", *COLAB_EXTRAS,
    ], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_ROOT)
], check=True)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
import torch
import yaml

from blackbox_detection.utils import load_checkpoint, seed_everything

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATASET_ROOT = DRIVE_PROJECT_ROOT / "DATASET"
DLC_ROOT = DATASET_ROOT / "DLC-2021"
DLC_SPLIT_CSV = DLC_ROOT / "dlc_split.csv"
OUTPUT_ROOT = DRIVE_PROJECT_ROOT / "outputs" / "stage1"
CONFIG_DIR = REPO_ROOT / "configs" / "stage1"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    required = {"DLC_ROOT": DLC_ROOT, "DLC_SPLIT_CSV": DLC_SPLIT_CSV}
    missing = [f"{k}: {v}" for k, v in required.items() if not v.exists()]
    if missing:
        raise FileNotFoundError("Missing required Drive paths:\n  " + "\n  ".join(missing))

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print("repo   :", REPO_ROOT)
print("branch :", BRANCH)
print("commit :", GIT_COMMIT)
print("torch  :", torch.__version__)
print("cuda   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu    :", torch.cuda.get_device_name(0))


## 2. V-JEPA 2.1-B source/config


In [ ]:
from blackbox_detection.stage1.dataset import Stage1VideoDataset, build_dataloader, video_batch_adapter
from blackbox_detection.stage1.evaluator import (
    AggregationConfig, Stage1Evaluator, aggregate_unit_predictions,
    evaluate_predictions, probabilities_to_labels, save_predictions,
)
from blackbox_detection.stage1.models import build_stage1_model, count_parameters
from blackbox_detection.stage1.sampling import build_clip_sampler
from blackbox_detection.stage1.transforms import ClipAugmentConfig, build_video_transforms

MODEL_NAME = "vjepa2_1_b"
CONFIG = yaml.safe_load((CONFIG_DIR / "vjepa2_1_b.yaml").read_text(encoding="utf-8"))
assert CONFIG["model"]["name"] == MODEL_NAME
SEED = int(CONFIG["train"]["seed"])
seed_everything(SEED, deterministic=False)

VJEPA_SOURCE_ROOT = Path("/content/vjepa2")
VJEPA_CKPT_DIR = DRIVE_PROJECT_ROOT / "pretrained" / "vjepa2"
VJEPA_CKPT_DIR.mkdir(parents=True, exist_ok=True)
VJEPA_CKPT = VJEPA_CKPT_DIR / "vjepa2_1_vitb_dist_vitG_384.pt"
if not (VJEPA_SOURCE_ROOT / ".git").is_dir():
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/facebookresearch/vjepa2.git",
        str(VJEPA_SOURCE_ROOT),
    ], check=True)
if not VJEPA_CKPT.is_file():
    subprocess.run([
        "wget", "-O", str(VJEPA_CKPT),
        "https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt",
    ], check=True)
params = dict(CONFIG["model"]["params"])
params["source_root"] = str(VJEPA_SOURCE_ROOT)
params["checkpoint_path"] = str(VJEPA_CKPT)
params["allow_download"] = False
print("V-JEPA source :", params["source_root"])
print("V-JEPA ckpt   :", params["checkpoint_path"])
print("arch          :", params["arch"])
print("runtime input :", params["input_frames"], "frames @", params["input_size"])


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from blackbox_detection.stage1.trainer import Stage1Trainer, TrainConfig

RUN_DIR = OUTPUT_ROOT / "09b_grouped_probe_cv"
RUN_DIR.mkdir(parents=True, exist_ok=True)
N_FOLDS = 5
FORCE_RETRAIN = False
EPOCH_OVERRIDE = None  # e.g. 6 for a pilot; None keeps the repository config
NUM_WORKERS = 2
print("output:", RUN_DIR)
print("folds :", N_FOLDS)


## 3. DLC train and untouched fixed val manifests


In [ ]:
VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv", ".m4v", ".webm"}

def _normalize_rel_text(value: str) -> str:
    return str(value).replace("\\", "/").strip().lstrip("./")

def _build_dlc_video_index() -> dict[tuple[str, str], str]:
    index = {}
    counts = {}
    for source in ("or", "re"):
        clips_root = DLC_ROOT / source / "clips_video"
        if not clips_root.is_dir():
            raise FileNotFoundError(f"DLC clips directory not found: {clips_root}")
        count = 0
        for path in clips_root.rglob("*"):
            if not path.is_file() or path.suffix.lower() not in VIDEO_EXTENSIONS:
                continue
            rel = path.relative_to(clips_root).with_suffix("").as_posix()
            key = (source, _normalize_rel_text(rel))
            if key in index and index[key] != str(path):
                raise ValueError(f"Duplicate DLC video key: {key}")
            index[key] = str(path)
            count += 1
        counts[source] = count
    print("indexed DLC videos:", counts)
    return index

DLC_VIDEO_INDEX = _build_dlc_video_index()

def _resolve_dlc_video(source: str, clip_id: str) -> str:
    source = str(source).strip().lower()
    clip_id = _normalize_rel_text(clip_id)
    key = (source, clip_id)
    if key in DLC_VIDEO_INDEX:
        return DLC_VIDEO_INDEX[key]
    matches = []
    for (src, rel), path in DLC_VIDEO_INDEX.items():
        if src != source:
            continue
        if rel.startswith(clip_id + "/") or rel.endswith("/" + clip_id) or rel == clip_id:
            matches.append(path)
    if len(matches) == 1:
        return matches[0]
    leaf = Path(clip_id).name
    matches = [
        path for (src, rel), path in DLC_VIDEO_INDEX.items()
        if src == source and Path(rel).name == leaf
    ]
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(f"Cannot resolve DLC source={source!r}, clip_id={clip_id!r}")

def load_dlc_manifest(split_name: str) -> pd.DataFrame:
    raw = pd.read_csv(DLC_SPLIT_CSV).copy()
    required = {
        "clip_id", "class", "source", "document_type", "document_id",
        "group", "split", "device", "condition",
    }
    missing = sorted(required - set(raw.columns))
    if missing:
        raise ValueError(f"dlc_split.csv missing columns: {missing}")
    raw["split"] = raw["split"].astype(str).str.strip().str.lower()
    raw["source"] = raw["source"].astype(str).str.strip().str.lower()
    raw["class"] = raw["class"].astype(str).str.strip().str.lower()
    frame = raw.loc[raw["split"].eq(split_name)].copy()
    if frame.empty:
        raise ValueError(f"No DLC rows for split={split_name!r}")
    frame["label"] = frame["class"].map({"original": "ORIGINAL", "rerecorded": "RERECORDED"})
    if frame["label"].isna().any():
        raise ValueError("Unexpected DLC class value found.")
    frame["video_id"] = "dlc__" + frame["clip_id"].astype(str).str.replace("/", "__", regex=False)
    frame["dataset"] = "dlc2021"
    frame["scene_type"] = "document"
    frame["video_path"] = [
        _resolve_dlc_video(source, clip_id)
        for source, clip_id in zip(frame["source"], frame["clip_id"])
    ]
    keep = [
        "video_path", "label", "video_id", "dataset", "scene_type",
        "clip_id", "source", "document_type", "document_id", "group",
        "device", "condition",
    ]
    out = frame[keep].reset_index(drop=True)
    missing_paths = [p for p in out["video_path"] if not Path(p).is_file()]
    if missing_paths:
        raise FileNotFoundError(
            f"{split_name}: {len(missing_paths)} missing videos; examples={missing_paths[:3]}"
        )
    return out


In [ ]:
train_df = load_dlc_manifest("train")
fixed_val_df = load_dlc_manifest("val")
print("train:", len(train_df), train_df["label"].value_counts().to_dict())
print("val  :", len(fixed_val_df), fixed_val_df["label"].value_counts().to_dict())
if train_df["group"].isna().any():
    raise ValueError("DLC train contains missing group values.")


## 4. Deterministic group folds


In [ ]:
label_to_int = {"ORIGINAL": 0, "RERECORDED": 1}
y = train_df["label"].map(label_to_int).to_numpy()
groups = train_df["group"].astype(str).to_numpy()
splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_id = np.full(len(train_df), -1, dtype=np.int64)
for fold, (_, va_idx) in enumerate(splitter.split(train_df, y, groups)):
    fold_id[va_idx] = fold
if (fold_id < 0).any():
    raise RuntimeError("Some rows did not receive a fold.")
fold_assignments = train_df[[
    "video_id", "label", "group", "document_id", "device", "condition", "document_type"
]].copy()
fold_assignments["fold"] = fold_id
fold_assignments.to_csv(RUN_DIR / "fold_assignments.csv", index=False)
for fold in range(N_FOLDS):
    tr = train_df.loc[fold_id != fold]
    va = train_df.loc[fold_id == fold]
    overlap = set(tr["group"].astype(str)) & set(va["group"].astype(str))
    if overlap:
        raise RuntimeError(f"fold {fold}: group leakage: {list(overlap)[:3]}")
    if va["label"].nunique() != 2:
        raise RuntimeError(f"fold {fold}: validation has one class only")
    print(
        f"fold {fold}: train={len(tr)} val={len(va)} "
        f"labels={va['label'].value_counts().to_dict()} groups={va['group'].nunique()}"
    )
display(pd.crosstab(fold_assignments["fold"], fold_assignments["label"], margins=True))


## 5. Shared transforms and loader builder


In [ ]:
video_cfg = CONFIG["data"]
aug_cfg = CONFIG["augmentation"]
tmp = build_stage1_model(MODEL_NAME, finetune_mode="head_only", unfreeze_last_n=0, **params)
pre = tmp.preprocessing()
del tmp
gc.collect()
train_transform, val_transform = build_video_transforms(
    crop_size=int(pre["input_size"]),
    mean=tuple(pre["mean"]),
    std=tuple(pre["std"]),
    train_config=ClipAugmentConfig(
        crop_size=int(pre["input_size"]),
        scale_range=tuple(aug_cfg["scale_range"]),
        ratio_range=tuple(aug_cfg["ratio_range"]),
        hflip_prob=float(aug_cfg["hflip_prob"]),
        brightness=float(aug_cfg["brightness"]),
        contrast=float(aug_cfg["contrast"]),
        perspective_prob=float(aug_cfg["perspective_prob"]),
        perspective_scale=float(aug_cfg["perspective_scale"]),
    ),
)

def make_video_loader(frame: pd.DataFrame, *, train: bool, seed: int):
    dataset = Stage1VideoDataset(
        frame.reset_index(drop=True),
        clip_sampler=build_clip_sampler(
            train=train,
            num_frames=int(video_cfg["num_frames"]),
            strides=tuple(video_cfg["train_strides"]),
            val_stride=int(video_cfg["val_stride"]),
            num_clips=1,
        ),
        transform=train_transform if train else val_transform,
        on_error="zero",
        deterministic=not train,
    )
    return build_dataloader(
        dataset,
        batch_size=int(video_cfg["batch_size"] if train else video_cfg["val_batch_size"]),
        shuffle=train,
        num_workers=NUM_WORKERS,
        seed=seed,
        drop_last=False,
        persistent_workers=NUM_WORKERS > 0,
    )

ADAPTER = video_batch_adapter()


## 6. Train/reuse one fresh attentive probe per fold


In [ ]:
train_cfg = dict(CONFIG["train"])
if EPOCH_OVERRIDE is not None:
    train_cfg["epochs"] = int(EPOCH_OVERRIDE)

oof_frames = []
fixed_val_frames = []
fold_summaries = []

for fold in range(N_FOLDS):
    print("\n" + "=" * 80)
    print("FOLD", fold)
    print("=" * 80)
    fold_dir = RUN_DIR / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    tr_df = train_df.loc[fold_id != fold].reset_index(drop=True)
    va_df = train_df.loc[fold_id == fold].reset_index(drop=True)
    train_loader = make_video_loader(tr_df, train=True, seed=SEED + fold)
    fold_val_loader = make_video_loader(va_df, train=False, seed=SEED + fold)
    fixed_val_loader = make_video_loader(fixed_val_df, train=False, seed=SEED)

    seed_everything(SEED + fold, deterministic=False)
    model = build_stage1_model(
        MODEL_NAME, finetune_mode="head_only", unfreeze_last_n=0, **params
    )
    best_ckpt = fold_dir / "best.pt"
    if FORCE_RETRAIN or not best_ckpt.is_file():
        trainer_cfg = TrainConfig(
            epochs=int(train_cfg["epochs"]),
            learning_rate=float(train_cfg["learning_rate"]),
            head_learning_rate=float(train_cfg["head_learning_rate"]),
            weight_decay=float(train_cfg["weight_decay"]),
            warmup_ratio=float(train_cfg["warmup_ratio"]),
            grad_accum_steps=int(train_cfg["grad_accum_steps"]),
            max_grad_norm=float(train_cfg["max_grad_norm"]),
            amp=bool(train_cfg["amp"]),
            label_smoothing=float(train_cfg.get("label_smoothing", 0.0)),
            early_stopping_patience=int(train_cfg["early_stopping_patience"]),
            eval_every=int(train_cfg["eval_every"]),
            seed=SEED + fold,
            output_dir=fold_dir,
            model_name=f"{MODEL_NAME}_09b_fold{fold}",
            wandb_enabled=False,
        )
        trainer = Stage1Trainer(
            model, trainer_cfg, adapter=ADAPTER,
            aggregation=AggregationConfig(frame_method="mean", video_method="mean"),
            model_config={
                "name": MODEL_NAME,
                "params": params,
                "experiment": "09b_grouped_probe_cv",
                "fold": fold,
            },
        )
        outcome = trainer.fit(train_loader, fold_val_loader)
        print(
            f"fold {fold}: best epoch={outcome.best_epoch} "
            f"F1={outcome.best_macro_f1:.4f} thr={outcome.best_threshold:.4f}"
        )
        del trainer, outcome

    load_checkpoint(best_ckpt, model=model, map_location="cpu", restore_rng_state=False)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device).eval()
    evaluator = Stage1Evaluator(
        model, ADAPTER, device=device, amp=False,
        aggregation=AggregationConfig(frame_method="mean", video_method="mean"),
    )
    oof_result = evaluator.evaluate(fold_val_loader, threshold=0.5, search_threshold=False)
    oof_pred = oof_result.predictions.copy(); oof_pred["fold"] = fold
    oof_frames.append(oof_pred)
    fixed_result = evaluator.evaluate(fixed_val_loader, threshold=0.5, search_threshold=False)
    fixed_pred = fixed_result.predictions.copy(); fixed_pred["fold"] = fold
    fixed_val_frames.append(fixed_pred)
    fold_summary = {
        "fold": fold,
        "oof_macro_f1_at_0.5": float(oof_result.macro_f1_at_default),
        "fixed_val_macro_f1_at_0.5": float(fixed_result.macro_f1_at_default),
        "num_train": int(len(tr_df)),
        "num_oof": int(len(va_df)),
    }
    fold_summaries.append(fold_summary)
    (fold_dir / "09b_eval_summary.json").write_text(
        json.dumps(fold_summary, indent=2), encoding="utf-8"
    )
    save_predictions(oof_pred, fold_dir / "oof_predictions.csv")
    save_predictions(fixed_pred, fold_dir / "fixed_val_predictions.csv")
    del model, evaluator, train_loader, fold_val_loader, fixed_val_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

display(pd.DataFrame(fold_summaries))


## 7. Materialise true OOF predictions


In [ ]:
oof = pd.concat(oof_frames, ignore_index=True)
if oof["video_id"].duplicated().any():
    raise RuntimeError("OOF contains duplicate video_id rows.")
if set(oof["video_id"]) != set(train_df["video_id"]):
    raise RuntimeError("OOF coverage does not match the DLC train split.")
oof_result = evaluate_predictions(oof, threshold=0.5, search_threshold=False)
save_predictions(oof_result.predictions, RUN_DIR / "oof_predictions.csv")
print("OOF Macro-F1 @0.5:", oof_result.macro_f1_at_default)
print("OOF per-class F1  :", oof_result.per_class_f1)


## 8. Fold-ensemble on untouched fixed validation


In [ ]:
stacked = pd.concat(fixed_val_frames, ignore_index=True)
ensemble = (
    stacked.groupby(["video_id", "label", "dataset"], as_index=False)
    .agg(prob_rerecorded=("prob_rerecorded", "mean"), fold_std=("prob_rerecorded", "std"))
)
ensemble["fold_std"] = ensemble["fold_std"].fillna(0.0)
ensemble["prob_original"] = 1.0 - ensemble["prob_rerecorded"]
ensemble_result = evaluate_predictions(ensemble, threshold=0.5, search_threshold=False)
save_predictions(ensemble_result.predictions, RUN_DIR / "val_predictions.csv")
print("fixed val ensemble F1 @0.5:", ensemble_result.macro_f1_at_default)
print("per-class                    :", ensemble_result.per_class_f1)
print("mean fold probability std    :", float(ensemble["fold_std"].mean()))
print("max fold probability std     :", float(ensemble["fold_std"].max()))


## 9. Save summary


In [ ]:
summary = {
    "experiment": "09b_grouped_probe_cv",
    "git_commit": GIT_COMMIT,
    "model": MODEL_NAME,
    "n_folds": N_FOLDS,
    "group_column": "group",
    "real_dlc_only": True,
    "a7_probe_warm_start": False,
    "encoder_pretrained_checkpoint": str(VJEPA_CKPT),
    "oof_macro_f1_at_0.5": float(oof_result.macro_f1_at_default),
    "fixed_val_ensemble_macro_f1_at_0.5": float(ensemble_result.macro_f1_at_default),
    "folds": fold_summaries,
}
(RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("saved to:", RUN_DIR)


## Decision rule

The fixed 104-video val may remain saturated. The important new quantity is
**grouped OOF Macro-F1**: each OOF prediction comes from a probe that did not
train on that video's `group`. Notebook 11 uses these OOF probabilities as the
base-domain proxy for fusion selection.
